# Detecção de fraude em transações de cartão de crédito

**Desafio de Projeto — DIO**

Este notebook percorre o pipeline completo de um modelo de detecção de fraude: exploração dos dados, preparação, treino de três algoritmos, avaliação pelas métricas que importam, ajuste de limiar, busca de hiperparâmetros, interpretação com SHAP e comparação de estratégias de balanceamento.

## O problema

A base tem **284.807 transações**, das quais apenas **492 são fraudes (0,17%)**. As variáveis `V1` a `V28` passaram por PCA para proteger a privacidade; `Time` marca o momento da transação, `Amount` o valor e `Class` a resposta (1 = fraude, 0 = normal).

## Por que a acurácia engana

Um modelo que responde "não é fraude" para tudo acerta **99,83%** das transações e ainda assim não encontra nenhuma fraude. Por isso, a avaliação aqui foca na **classe de fraude**:

- **Recall:** quantas fraudes reais o modelo encontrou;
- **Precisão:** quantas das transações marcadas como fraude eram mesmo fraude;
- **F1:** equilíbrio entre precisão e recall;
- **PR AUC:** área sob a curva precisão-recall, mais informativa em bases desbalanceadas;
- **ROC AUC:** capacidade geral de separar as classes.

## 1. Configuração

Importamos as bibliotecas, definimos a semente aleatória para reprodutibilidade e criamos as pastas onde as figuras e as métricas serão salvas.

In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"

Path("images").mkdir(exist_ok=True)
Path("resultados").mkdir(exist_ok=True)

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 12

## 2. Exploração dos dados

O notebook lê o CSV **diretamente pela URL**, como no desafio. O arquivo tem cerca de 144 MB e não deve ser versionado no repositório.

In [ ]:
df = pd.read_csv(DATA_URL)

print(f"Transações: {df.shape[0]:,}".replace(",", "."))
print(f"Colunas: {df.shape[1]}")
df.head()

### Qualidade dos dados

Antes de modelar, verificamos valores nulos e linhas duplicadas. Duplicatas podem aparecer ao mesmo tempo no treino e no teste, inflando as métricas, então vamos removê-las e registrar o impacto.

In [ ]:
print(f"Valores nulos: {int(df.isna().sum().sum())}")
print(f"Linhas duplicadas: {int(df.duplicated().sum())}")
print(f"Fraudes entre as duplicadas: {int(df.loc[df.duplicated(), 'Class'].sum())}")

df[["Time", "Amount", "Class"]].describe().round(2)

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print(f"Base após remover duplicadas: {df.shape[0]:,} transações".replace(",", "."))

contagem = df["Class"].value_counts().sort_index()
percentual = df["Class"].value_counts(normalize=True).sort_index() * 100

tabela_classes = pd.DataFrame({
    "Classe": ["Normal (0)", "Fraude (1)"],
    "Transações": contagem.values,
    "Percentual (%)": percentual.round(4).values,
})
tabela_classes

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
barras = sns.barplot(
    data=tabela_classes,
    x="Classe",
    y="Transações",
    hue="Classe",
    legend=False,
    ax=ax,
)
ax.set_yscale("log")
ax.set_title("Distribuição das classes (escala logarítmica)")
ax.set_ylabel("Número de transações (log)")
for barra, valor in zip(barras.patches, tabela_classes["Transações"]):
    ax.annotate(
        f"{valor:,}".replace(",", "."),
        (barra.get_x() + barra.get_width() / 2, barra.get_height()),
        ha="center",
        va="bottom",
        fontsize=9,
    )
plt.tight_layout()
plt.savefig("images/01_distribuicao_classes.png", dpi=150, bbox_inches="tight")
plt.show()

### Amount por classe

A distribuição de valores é muito assimétrica. Por isso, aplicamos `np.log1p` no valor antes de padronizar: o log comprime os valores altos e deixa a escala mais próxima de uma distribuição normal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for classe, rotulo, cor in [(0, "Normal", "#4c72b0"), (1, "Fraude", "#c44e52")]:
    valores = np.log1p(df.loc[df["Class"] == classe, "Amount"])
    sns.histplot(
        valores,
        bins=60,
        stat="density",
        element="step",
        fill=False,
        label=rotulo,
        color=cor,
        ax=axes[0],
    )
axes[0].set_title("Distribuição de log1p(Amount) por classe")
axes[0].set_xlabel("log1p(Amount)")
axes[0].legend()

sns.boxplot(data=df, x="Class", y="Amount", ax=axes[1], showfliers=False)
axes[1].set_xticks([0, 1], ["Normal", "Fraude"])
axes[1].set_title("Amount por classe (sem outliers)")
axes[1].set_ylabel("Amount")

plt.tight_layout()
plt.savefig("images/02_amount_por_classe.png", dpi=150, bbox_inches="tight")
plt.show()

df.groupby("Class")["Amount"].describe().round(2)

### Fraudes ao longo do tempo

`Time` é o tempo em segundos desde a primeira transação da base, cobrindo cerca de 48 horas. A variável pode ajudar a capturar padrões temporais, mas não deve ser interpretada como uma data real.

In [ ]:
df["Hora"] = (df["Time"] / 3600) % 24

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(
    data=df,
    x="Hora",
    hue="Class",
    bins=48,
    stat="density",
    common_norm=False,
    element="step",
    fill=False,
    ax=axes[0],
)
axes[0].set_title("Distribuição por hora do dia")
axes[0].set_xlabel("Hora do dia")

fraudes_por_hora = df[df["Class"] == 1].groupby(df["Hora"].astype(int)).size()
axes[1].bar(fraudes_por_hora.index, fraudes_por_hora.values, color="#c44e52")
axes[1].set_title("Fraudes por hora do dia")
axes[1].set_xlabel("Hora do dia")
axes[1].set_ylabel("Quantidade de fraudes")

plt.tight_layout()
plt.savefig("images/03_fraudes_por_hora.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Preparação dos dados

Criamos `Amount_log` com `np.log1p` e mantemos `Time` e as 28 componentes principais. A padronização fica **dentro de um `Pipeline`**, ajustada apenas no treino, para evitar vazamento de informação do teste.

In [ ]:
df["Amount_log"] = np.log1p(df["Amount"])

features = [f"V{i}" for i in range(1, 29)] + ["Time", "Amount_log"]
X = df[features]
y = df["Class"]

print(f"{len(features)} variáveis preditoras")
print(features)

### Separação treino/teste

Usamos 80% para treino e 20% para teste, com `stratify=y` para manter a mesma proporção de fraudes nos dois conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

resumo_split = pd.DataFrame(
    {
        "Treino": [len(y_train), int(y_train.sum()), 100 * y_train.mean()],
        "Teste": [len(y_test), int(y_test.sum()), 100 * y_test.mean()],
    },
    index=["Transações", "Fraudes", "Percentual de fraude (%)"],
).round(4)
resumo_split

In [ ]:
scaler_demo = StandardScaler()
X_train_scaled = scaler_demo.fit_transform(X_train)
X_test_scaled = scaler_demo.transform(X_test)

print("Média das 5 primeiras variáveis no treino padronizado:")
print(np.round(X_train_scaled[:, :5].mean(axis=0), 6))
print("Desvio-padrão das 5 primeiras variáveis no treino padronizado:")
print(np.round(X_train_scaled[:, :5].std(axis=0), 6))
print("Formatos:", X_train_scaled.shape, X_test_scaled.shape)

> O `StandardScaler` acima serve apenas para visualizar o efeito da padronização. Nos modelos, ele é ajustado dentro do `Pipeline`, em cada divisão da validação cruzada, para não usar informação do teste.

## 4. Modelos e avaliação

Vamos comparar três algoritmos:

1. **Regressão logística** como baseline, com `class_weight='balanced'`;
2. **Random Forest**, também com `class_weight='balanced'`;
3. **XGBoost**, usando `scale_pos_weight` para dar mais peso à fraude.

A função `avaliar_modelo` calcula as métricas da classe de fraude, imprime o relatório de classificação e mostra a matriz de confusão.

In [ ]:
resultados = []
probas = {}


def avaliar_modelo(nome, modelo, limiar=0.5, registrar=True, chave=None):
    chave = chave or nome
    proba = modelo.predict_proba(X_test)[:, 1]
    pred = (proba >= limiar).astype(int)
    probas[chave] = proba

    metricas = {
        "Modelo": nome,
        "Limiar": limiar,
        "Precisão": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, proba),
        "PR AUC": average_precision_score(y_test, proba),
    }

    print(f"=== {nome} | limiar = {limiar:.2f} ===")
    print(
        classification_report(
            y_test,
            pred,
            target_names=["Normal", "Fraude"],
            digits=4,
        )
    )

    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Normal", "Fraude"])
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(f"Matriz de confusão — {nome} (limiar {limiar:.2f})")
    plt.tight_layout()
    plt.show()

    if registrar:
        resultados.append(metricas)

    return metricas

### Baseline: regressão logística

In [ ]:
modelo_lr = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
modelo_lr.fit(X_train, y_train)
avaliar_modelo("Regressão Logística", modelo_lr)

### Random Forest

In [ ]:
modelo_rf = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
modelo_rf.fit(X_train, y_train)
avaliar_modelo("Random Forest", modelo_rf)

### XGBoost

O XGBoost não tem `class_weight`, então usamos `scale_pos_weight`: a razão entre o número de transações normais e de fraudes no treino.

In [ ]:
escala_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {escala_pos:.2f}")

modelo_xgb = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=5,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=escala_pos,
                eval_metric="aucpr",
                tree_method="hist",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
modelo_xgb.fit(X_train, y_train)
avaliar_modelo("XGBoost", modelo_xgb)

### Comparação dos modelos

A tabela abaixo ordena os modelos pelo **recall** da classe de fraude. O recall é a métrica mais importante aqui: deixar uma fraude passar costuma custar mais caro do que investigar um falso positivo.

In [ ]:
tabela_modelos = (
    pd.DataFrame(resultados)
    .set_index("Modelo")
    .sort_values("Recall", ascending=False)
)
tabela_modelos.round(4).to_csv("resultados/metricas_modelos.csv", index=True)
tabela_modelos.style.format("{:.4f}").background_gradient(
    cmap="Blues", subset=["Recall", "F1", "PR AUC"]
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
tabela_modelos[["Precisão", "Recall", "F1"]].plot(
    kind="bar",
    ax=ax,
    color=["#4c72b0", "#dd8452", "#55a868"],
)
ax.set_title("Comparação dos modelos na classe de fraude")
ax.set_ylabel("Métrica")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("images/04_comparacao_modelos.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
modelos_curva = ["Regressão Logística", "Random Forest", "XGBoost"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for nome in modelos_curva:
    proba = probas[nome]
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(
        fpr,
        tpr,
        label=f"{nome} (AUC = {roc_auc_score(y_test, proba):.3f})",
    )

    precisao, recall, _ = precision_recall_curve(y_test, proba)
    axes[1].plot(
        recall,
        precisao,
        label=f"{nome} (AP = {average_precision_score(y_test, proba):.3f})",
    )

axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aleatório")
axes[0].set_title("Curva ROC")
axes[0].set_xlabel("Taxa de falsos positivos")
axes[0].set_ylabel("Taxa de verdadeiros positivos")
axes[0].legend()

axes[1].set_title("Curva precisão-recall")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precisão")
axes[1].legend()

plt.tight_layout()
plt.savefig("images/05_curvas_roc_pr.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Ajuste do limiar de decisão

Por padrão, o modelo marca como fraude quando a probabilidade é maior ou igual a **0,5**. Em fraude, é comum testar um limiar mais baixo, como **0,3**, para tentar capturar mais fraudes. O efeito, porém, depende do modelo: o recall pode subir, mas a precisão costuma cair, porque mais transações normais passam a ser investigadas.

Vamos comparar o trade-off no **XGBoost** e no **Random Forest**, que têm distribuições de probabilidade diferentes.

In [ ]:
def tabela_por_limiar(proba):
    linhas = []
    for limiar in np.arange(0.05, 0.96, 0.05):
        pred = (proba >= limiar).astype(int)
        linhas.append(
            {
                "Limiar": limiar,
                "Precisão": precision_score(y_test, pred, zero_division=0),
                "Recall": recall_score(y_test, pred, zero_division=0),
                "F1": f1_score(y_test, pred, zero_division=0),
            }
        )
    return pd.DataFrame(linhas)


tabela_limiares_xgb = tabela_por_limiar(probas["XGBoost"])
tabela_limiares_rf = tabela_por_limiar(probas["Random Forest"])

tabela_limiares = pd.concat(
    [
        tabela_limiares_xgb.assign(Modelo="XGBoost"),
        tabela_limiares_rf.assign(Modelo="Random Forest"),
    ],
    ignore_index=True,
)
tabela_limiares.round(4).to_csv("resultados/metricas_limiares.csv", index=False)

display(tabela_limiares_xgb.round(4))
display(tabela_limiares_rf.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

for ax, tabela, titulo in [
    (axes[0], tabela_limiares_xgb, "XGBoost"),
    (axes[1], tabela_limiares_rf, "Random Forest"),
]:
    for metrica, cor in [
        ("Precisão", "#4c72b0"),
        ("Recall", "#dd8452"),
        ("F1", "#55a868"),
    ]:
        ax.plot(
            tabela["Limiar"],
            tabela[metrica],
            marker="o",
            markersize=3,
            label=metrica,
            color=cor,
        )
    ax.axvline(0.3, color="red", linestyle="--", label="Limiar 0,3")
    ax.axvline(0.5, color="gray", linestyle=":", label="Limiar 0,5")
    ax.set_title(f"Trade-off por limiar — {titulo}")
    ax.set_xlabel("Limiar de decisão")
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Métrica")
plt.tight_layout()
plt.savefig("images/06_limiar_decisao.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
avaliar_modelo(
    "XGBoost (limiar 0,3)",
    modelo_xgb,
    limiar=0.3,
    chave="XGBoost (0.3)",
)

avaliar_modelo(
    "Random Forest (limiar 0,3)",
    modelo_rf,
    limiar=0.3,
    chave="Random Forest (0.3)",
)

candidatos_limiar = [
    "XGBoost",
    "XGBoost (limiar 0,3)",
    "Random Forest",
    "Random Forest (limiar 0,3)",
]
tabela_candidatos_limiar = (
    pd.DataFrame([r for r in resultados if r["Modelo"] in candidatos_limiar])
    .set_index("Modelo")
    .loc[candidatos_limiar]
)
tabela_candidatos_limiar.round(4)

## 6. Busca de hiperparâmetros com `GridSearchCV`

Agora colocamos padronização e XGBoost em um `Pipeline` e testamos algumas combinações de hiperparâmetros com validação cruzada estratificada. O critério de escolha é o **recall**: queremos o modelo que encontra mais fraudes.

> O `GridSearchCV` usa o limiar padrão de 0,5 para calcular o recall. Depois, aplicamos o limiar de 0,3 ao melhor modelo encontrado.

In [ ]:
pipeline_busca = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "model",
            XGBClassifier(
                eval_metric="aucpr",
                scale_pos_weight=escala_pos,
                tree_method="hist",
                n_jobs=1,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

grade = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [3, 6],
}

busca = GridSearchCV(
    pipeline_busca,
    grade,
    scoring="recall",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=4,
    verbose=1,
)
busca.fit(X_train, y_train)

print("Melhores parâmetros:", busca.best_params_)
print(f"Melhor recall na validação cruzada: {busca.best_score_:.4f}")

modelo_busca = busca.best_estimator_
avaliar_modelo("XGBoost (GridSearchCV)", modelo_busca)

In [ ]:
avaliar_modelo(
    "XGBoost GridSearchCV (limiar 0,3)",
    modelo_busca,
    limiar=0.3,
    chave="XGBoost GridSearchCV (0.3)",
)

candidatos_finais = [
    "XGBoost",
    "XGBoost (limiar 0,3)",
    "Random Forest",
    "Random Forest (limiar 0,3)",
    "XGBoost (GridSearchCV)",
    "XGBoost GridSearchCV (limiar 0,3)",
]
tabela_finais = (
    pd.DataFrame([r for r in resultados if r["Modelo"] in candidatos_finais])
    .set_index("Modelo")
    .loc[candidatos_finais]
    .sort_values("Recall", ascending=False)
)
tabela_finais.round(4).to_csv("resultados/metricas_finais.csv", index=True)
tabela_finais.style.format("{:.4f}").background_gradient(
    cmap="Blues", subset=["Recall", "F1", "PR AUC"]
)

## 7. Importância das variáveis e SHAP

A importância nativa do XGBoost mostra quais variáveis o modelo mais usou nas divisões das árvores. O SHAP vai além: mostra **quanto** cada variável empurrou uma predição específica para fraude ou para normal.

In [ ]:
importancias = pd.Series(
    modelo_xgb.named_steps["model"].feature_importances_,
    index=features,
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    x=importancias.head(15).values,
    y=importancias.head(15).index,
    ax=ax,
    color="#4c72b0",
)
ax.set_title("15 variáveis mais importantes — XGBoost")
ax.set_xlabel("Importância")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("images/07_importancia_variaveis.png", dpi=150, bbox_inches="tight")
plt.show()

importancias.head(15).round(4)

In [ ]:
import shap

amostra = X_test.sample(n=min(2000, len(X_test)), random_state=RANDOM_STATE)
scaler_xgb = modelo_xgb.named_steps["scaler"]
amostra_scaled = pd.DataFrame(
    scaler_xgb.transform(amostra),
    columns=features,
    index=amostra.index,
)

explainer = shap.TreeExplainer(modelo_xgb.named_steps["model"])
shap_values = explainer.shap_values(amostra_scaled)
print("Formato dos valores SHAP:", np.shape(shap_values))

shap.summary_plot(shap_values, amostra_scaled, max_display=15, show=False)
plt.title("SHAP — impacto das variáveis na predição de fraude")
plt.tight_layout()
plt.savefig("images/08_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

### Explicando uma fraude específica

O gráfico de waterfall mostra, para uma transação real classificada como fraude, quais variáveis aumentaram ou diminuíram a probabilidade prevista.

In [ ]:
proba_amostra = modelo_xgb.predict_proba(amostra)[:, 1]
rotulo_amostra = y_test.loc[amostra.index]
candidatos = np.where((rotulo_amostra == 1) & (proba_amostra >= 0.5))[0]
if len(candidatos) == 0:
    candidatos = np.where(rotulo_amostra == 1)[0]

i = int(candidatos[0])
explicacao = shap.Explanation(
    values=shap_values[i],
    base_values=explainer.expected_value,
    data=amostra_scaled.iloc[i].values,
    feature_names=features,
)

print(
    f"Transação real: {'fraude' if rotulo_amostra.iloc[i] == 1 else 'normal'} "
    f"| probabilidade prevista: {proba_amostra[i]:.4f}"
)
shap.plots.waterfall(explicacao, max_display=12, show=False)
plt.tight_layout()
plt.savefig("images/09_shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Undersampling e oversampling

As duas estratégias clássicas para desbalanceamento são:

- **Undersampling:** reduzir a classe majoritária;
- **Oversampling:** replicar ou gerar exemplos da classe minoritária.

A regra é testar: o resultado depende muito do dataset. Comparamos as duas com regressão logística, usando o mesmo conjunto de teste.

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler


def avaliar_amostragem(nome, sampler):
    pipeline = ImbPipeline(
        [
            ("scaler", StandardScaler()),
            ("sampler", sampler),
            (
                "model",
                LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            ),
        ]
    )
    pipeline.fit(X_train, y_train)
    proba = pipeline.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    return {
        "Estratégia": nome,
        "Precisão": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "PR AUC": average_precision_score(y_test, proba),
    }


pipeline_sem_balanceamento = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)
pipeline_sem_balanceamento.fit(X_train, y_train)
proba_sem = pipeline_sem_balanceamento.predict_proba(X_test)[:, 1]
pred_sem = (proba_sem >= 0.5).astype(int)

tabela_amostragem = pd.DataFrame(
    [
        {
            "Estratégia": "Sem balanceamento",
            "Precisão": precision_score(y_test, pred_sem, zero_division=0),
            "Recall": recall_score(y_test, pred_sem, zero_division=0),
            "F1": f1_score(y_test, pred_sem, zero_division=0),
            "PR AUC": average_precision_score(y_test, proba_sem),
        },
        avaliar_amostragem(
            "Undersampling", RandomUnderSampler(random_state=RANDOM_STATE)
        ),
        avaliar_amostragem(
            "Oversampling", RandomOverSampler(random_state=RANDOM_STATE)
        ),
    ]
)

tabela_amostragem.round(4).to_csv(
    "resultados/metricas_amostragem.csv", index=False
)
tabela_amostragem.style.format(
    "{:.4f}", subset=["Precisão", "Recall", "F1", "PR AUC"]
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
tabela_amostragem.set_index("Estratégia")[["Precisão", "Recall", "F1"]].plot(
    kind="bar",
    ax=ax,
    color=["#4c72b0", "#dd8452", "#55a868"],
)
ax.set_title("Efeito do balanceamento no recall da classe de fraude")
ax.set_ylabel("Métrica")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("images/10_amostragem.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Conclusões

- O desbalanceamento muda a pergunta: não é "quanto o modelo acerta?", e sim "quantas fraudes ele encontra e a que custo?".
- A remoção de 1.081 duplicatas deixou 283.726 transações e 473 fraudes (0,1667%), evitando que a mesma transação aparecesse no treino e no teste.
- No limiar 0,5, o XGBoost teve o melhor equilíbrio entre precisão e recall (precisão 0,8556, recall 0,8105, F1 0,8324). O Random Forest foi mais preciso (0,9583), mas encontrou menos fraudes (recall 0,7263). A regressão logística encontrou muitas fraudes (recall 0,8737), mas com precisão de apenas 0,0554.
- Baixar o limiar para 0,3 não ajudou o XGBoost: o recall ficou em 0,8105 e a precisão caiu para 0,7624. Já no Random Forest, o limiar 0,3 subiu o recall de 0,7263 para 0,7895 e o F1 de 0,8263 para 0,8475.
- O XGBoost ajustado pelo `GridSearchCV` (`max_depth=3`, `n_estimators=200`) com limiar 0,3 foi o candidato com maior recall entre os modelos com precisão razoável: precisão 0,8404, recall 0,8316, F1 0,8360 e PR AUC 0,8345. É o modelo escolhido.
- O `GridSearchCV` otimiza o recall usando o limiar padrão de 0,5; uma evolução é usar um scorer customizado no limiar escolhido.
- O SHAP mostra que `V14`, `V4`, `V12`, `V10` e `V11` foram as variáveis de maior impacto. Na transação de fraude explicada, `V14`, `V12`, `V10` e `V4` empurraram a predição para fraude, enquanto `V8` e `V13` puxaram para o lado normal.
- Undersampling e oversampling aumentaram o recall da regressão logística, mas com precisão muito baixa. O melhor caminho foi o XGBoost com peso de classe e ajuste de limiar.

In [ ]:
melhor_recall = tabela_modelos["Recall"].idxmax()
melhor_f1 = tabela_modelos["F1"].idxmax()
modelo_final_nome = "XGBoost GridSearchCV (limiar 0,3)"
modelo_final = tabela_finais.loc[modelo_final_nome]

print(f"Modelo escolhido: {modelo_final_nome}")
print(f"  Precisão: {modelo_final['Precisão']:.4f}")
print(f"  Recall: {modelo_final['Recall']:.4f}")
print(f"  F1: {modelo_final['F1']:.4f}")
print(f"  PR AUC: {modelo_final['PR AUC']:.4f}")
print()
print(f"Maior recall no limiar 0,5: {melhor_recall}")
print(f"  Recall: {tabela_modelos.loc[melhor_recall, 'Recall']:.4f}")
print(f"  Precisão: {tabela_modelos.loc[melhor_recall, 'Precisão']:.4f}")
print(f"  F1: {tabela_modelos.loc[melhor_recall, 'F1']:.4f}")
print()
print(f"Maior F1 no limiar 0,5: {melhor_f1}")
print(f"  Precisão: {tabela_modelos.loc[melhor_f1, 'Precisão']:.4f}")
print(f"  Recall: {tabela_modelos.loc[melhor_f1, 'Recall']:.4f}")
print(f"  F1: {tabela_modelos.loc[melhor_f1, 'F1']:.4f}")